# 📦 Лабораторная работа №7. Вариант 29: Генератор маркетинговых планов
**Шаг 0:** Перед запуском убедитесь, что в левой панели Colab (раздел 🔑 `Секреты`) создан ключ:
- `Name`: `GOOGLE_API_KEY`
- `Value`: ваш ключ от `aistudio.google.com`
- ✅ Включен тумблер `Notebook access`

In [ ]:
!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata
import json
import time

try:
    api_key = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=api_key)
    print("✅ API-ключ успешно загружен.")
except Exception as e:
    print(f"❌ Ошибка загрузки ключа: {e}. Проверьте раздел 'Секреты' в Colab.")

## 🏗 Задание 1: Класс PromptBuilder
Отвечает только за динамическую сборку промпта. Не обращается к API.

In [ ]:
class MarketingPlanPromptBuilder:
    def __init__(self):
        self.system_role = (
            "Ты — опытный маркетинговый директор и эксперт по стратегическому планированию. "
            "Твоя задача — создавать структурированные, реалистичные и измеримые маркетинговые планы."
        )
        self.product = None
        self.budget = None
        self.timeline = None

    def set_product(self, product):
        self.product = product
        return self

    def set_budget(self, budget):
        self.budget = budget
        return self

    def set_timeline(self, timeline):
        self.timeline = timeline
        return self

    def build_prompt(self):
        if not self.product or not self.budget or not self.timeline:
            raise ValueError("Необходимо задать продукт, бюджет и сроки.")

        instruction = (
            f"Разработай маркетинговый план для продукта: '{self.product}'.\n"
            f"Бюджет: {self.budget}.\n"
            f"Сроки реализации: {self.timeline}.\n\n"
            "Требования к ответу:\n"
            "1. Верни ответ ТОЛЬКО в формате валидного JSON без markdown-оберток и пояснений.\n"
            "2. Строго следуй схеме:\n"
            "{\n"
            '  "title": "Название плана",\n'
            '  "slides": [\n'
            '    {"slide": 1, "topic": "Каналы продвижения", "points": ["канал1", "канал2"], "speaker_notes": "Комментарий"},\n'
            '    {"slide": 2, "topic": "Медиаплан", "points": ["распределение бюджета по месяцам"], "speaker_notes": "Комментарий"},\n'
            '    {"slide": 3, "topic": "Ожидаемые показатели (KPI)", "points": ["метрика1", "метрика2"], "speaker_notes": "Комментарий"},\n'
            '    {"slide": 4, "topic": "Креативная концепция", "points": ["идея", "визуал", "тон"], "speaker_notes": "Комментарий"}\n'
            '  ]\n'
            "}\n"
            "3. Подбирай каналы и KPI строго в рамках указанного бюджета и сроков."
        )

        return [
            {"role": "system", "content": self.system_role},
            {"role": "user", "content": instruction}
        ]

## 🧪 Тест сборки промпта (Скриншот для `1.png`)
Запустите ячейку и сделайте скриншот вывода консоли.

In [ ]:
builder = MarketingPlanPromptBuilder()
prompt_data = builder.set_product("Умные часы FitLife").set_budget("500 000 ₽").set_timeline("2 месяца").build_prompt()

print("--- СБОРКА ПРОМПТА (ЗАДАНИЕ 1) ---")
print(f"System Role: {prompt_data[0]['content'][:50]}...")
print(f"User Prompt:\n{prompt_data[1]['content']}")
print("\n✅ Проверка структуры завершена. Промпт готов к отправке в LLM.")

## 🌐 Задание 2: API-движок и Pipeline
Реализует отправку запроса, retry-логику, очистку JSON и сохранение файлов.

In [ ]:
class LLMMarketingEngine:
    def __init__(self):
        self.model = genai.GenerativeModel('gemini-1.5-flash')

    def generate(self, messages):
        full_prompt = f"{messages[0]['content']}\n\n{messages[1]['content']}"
        generation_config = genai.types.GenerationConfig(
            temperature=0.3,
            top_p=0.9,
            candidate_count=1
        )
        response = self.model.generate_content(full_prompt, generation_config=generation_config)
        return response.text

def clean_and_parse_json(raw_text):
    start = raw_text.find('{')
    end = raw_text.rfind('}') + 1
    if start == -1 or end == 0:
        raise ValueError("JSON не найден в ответе модели.")
    return json.loads(raw_text[start:end])

def safe_generate_with_retry(engine, messages, retries=3):
    for attempt in range(retries):
        try:
            raw = engine.generate(messages)
            return clean_and_parse_json(raw)
        except Exception as e:
            print(f"[Attempt {attempt+1}] Ошибка: {e}")
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
            else:
                raise

def run_pipeline(product, budget, timeline):
    builder = MarketingPlanPromptBuilder()
    engine = LLMMarketingEngine()
    
    print(f"\n🚀 Генерация плана для: {product} (Бюджет: {budget})")
    messages = builder.set_product(product).set_budget(budget).set_timeline(timeline).build_prompt()
    
    plan_json = safe_generate_with_retry(engine, messages)
    
    safe_name = product.replace(' ', '_').lower().replace("'", "").replace("\"", "")
    filename_json = f"plan_{safe_name}.json"
    with open(filename_json, 'w', encoding='utf-8') as f:
        json.dump(plan_json, f, ensure_ascii=False, indent=2)
        
    filename_txt = filename_json.replace('.json', '.txt')
    with open(filename_txt, 'w', encoding='utf-8') as txt:
        txt.write(f"МАРКЕТИНГОВЫЙ ПЛАН: {plan_json['title']}\n")
        txt.write("="*50 + "\n")
        for slide in plan_json['slides']:
            txt.write(f"\n[СЛАЙД {slide['slide']}] {slide['topic']}\n")
            txt.write("• " + "\n• ".join(slide['points']) + "\n")
            txt.write(f"Примечание спикера: {slide['speaker_notes']}\n")
            
    print(f"✅ Успешно сохранено в {filename_json} и {filename_txt}")
    return plan_json

## 🚀 Задание 3: Тестовые запуски
Запустите ячейку. Сделайте скриншоты:
- Вывод консоли (`2.png`)
- Откройте файл `plan_*.json` в панели файлов слева → скриншот (`3.png`)
- Откройте файл `plan_*.txt` → скриншот (`4.png`)

In [ ]:
TEST_CASES = [
    {"product": "Мобильное приложение для йоги", "budget": "300 000 ₽", "timeline": "2 месяца"},
    {"product": "SaaS-платформа для HR", "budget": "1 500 000 ₽", "timeline": "3 месяца"},
    {"product": "Органическая косметика EcoSkin", "budget": "500 000 ₽", "timeline": "1 месяц"}
]

for i, case in enumerate(TEST_CASES, 1):
    print(f"\n--- ТЕСТ {i}/3 ---")
    try:
        run_pipeline(case["product"], case["budget"], case["timeline"])
        time.sleep(2)
    except Exception as e:
        print(f"❌ Ошибка в тесте {i}: {e}")

print("\n🎉 Все тесты завершены. Проверьте файлы в панели 'Файлы' слева.")

In [ ]:
# 📂 Быстрый просмотр первого сгенерированного файла для проверки
import glob
files = glob.glob("plan_*.json")
if files:
    print(f"📄 Содержимое {files[0]}:")
    with open(files[0], 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Файлы ещё не созданы. Запустите предыдущую ячейку.")